In [28]:
import pandas as pd 

flows = pd.DataFrame({
    "src_ip": ["192.168.56.11", "192.168.56.11", "10.0.0.5", "10.0.0.7"],
    "dst_port": [80, 22, 443, 22],
    "packets": [12, 950, 30, 4],
    "duration_s": [1.2, 0.8, 3.5, 0.1],
    "label": ["normal", "attack", "normal", "attack"],
})
flows

,src_ip,dst_port,packets,duration_s,label
0,192.168.56.11,80,12,1.2,normal
1,192.168.56.11,22,950,0.8,attack
2,10.0.0.5,443,30,3.5,normal
3,10.0.0.7,22,4,0.1,attack


In [29]:
print(flows.shape)
print(flows["label"].value_counts())
flows.describe()

(4, 5)
label
normal    2
attack    2
Name: count, dtype: int64


,dst_port,packets,duration_s
count,4.000000,4.000000,4.00000
mean,141.750000,249.000000,1.40000
std,202.685923,467.459802,1.47196
min,22.000000,4.000000,0.10000
25%,22.000000,10.000000,0.62500
50%,51.000000,21.000000,1.00000
75%,170.750000,260.000000,1.77500
max,443.000000,950.000000,3.50000


In [30]:
df = pd.read_csv("../data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
print(df.shape)

(286467, 79)


In [31]:
print(df.columns.tolist())

[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [32]:
df.columns = df.columns.str.strip()
print(df["Label"].value_counts())

Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [33]:
df.groupby("Label")[["Flow Duration", "Total Fwd Packets", "Destination Port"]].median()

,Flow Duration,Total Fwd Packets,Destination Port
Label,,,
BENIGN,30964.0,2.0,80.0
PortScan,47.0,1.0,3690.0


In [34]:
print(df.groupby("Label")["Destination Port"].nunique())
print(df[df["Label"] == "BENIGN"]["Destination Port"].value_counts().head(5))

Label
BENIGN      13694
PortScan     1000
Name: Destination Port, dtype: int64
Destination Port
53     58585
443    26695
80     16368
22       672
123      621
Name: count, dtype: int64


In [35]:
import numpy as np
print(df.isna().sum().sum())
print(np.isinf(df.select_dtypes(include="number")).sum().sum())

15
727


In [36]:
bad = df.replace([np.inf, -np.inf], np.nan).isna().sum()
print(bad[bad > 0])

Flow Bytes/s      371
Flow Packets/s    371
dtype: int64


In [37]:
df = df.replace([np.inf, -np.inf], np.nan).dropna()
print(df.shape)
print(df["Label"].value_counts())

(286096, 79)
Label
PortScan    158804
BENIGN      127292
Name: count, dtype: int64


In [38]:
from sklearn.model_selection import train_test_split

y = (df["Label"] == "PortScan").astype(int)
X = df.drop(columns=["Label"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(228876, 78) (57220, 78)
0.5550734895751411 0.5550681579867179


In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["BENIGN", "PortScan"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     25459
    PortScan       1.00      1.00      1.00     31761

    accuracy                           1.00     57220
   macro avg       1.00      1.00      1.00     57220
weighted avg       1.00      1.00      1.00     57220

[[25458     1]
 [    4 31757]]


In [40]:
for i, name in enumerate(X_train.columns):
    print(i, name)

0 Destination Port
1 Flow Duration
2 Total Fwd Packets
3 Total Backward Packets
4 Total Length of Fwd Packets
5 Total Length of Bwd Packets
6 Fwd Packet Length Max
7 Fwd Packet Length Min
8 Fwd Packet Length Mean
9 Fwd Packet Length Std
10 Bwd Packet Length Max
11 Bwd Packet Length Min
12 Bwd Packet Length Mean
13 Bwd Packet Length Std
14 Flow Bytes/s
15 Flow Packets/s
16 Flow IAT Mean
17 Flow IAT Std
18 Flow IAT Max
19 Flow IAT Min
20 Fwd IAT Total
21 Fwd IAT Mean
22 Fwd IAT Std
23 Fwd IAT Max
24 Fwd IAT Min
25 Bwd IAT Total
26 Bwd IAT Mean
27 Bwd IAT Std
28 Bwd IAT Max
29 Bwd IAT Min
30 Fwd PSH Flags
31 Bwd PSH Flags
32 Fwd URG Flags
33 Bwd URG Flags
34 Fwd Header Length
35 Bwd Header Length
36 Fwd Packets/s
37 Bwd Packets/s
38 Min Packet Length
39 Max Packet Length
40 Packet Length Mean
41 Packet Length Std
42 Packet Length Variance
43 FIN Flag Count
44 SYN Flag Count
45 RST Flag Count
46 PSH Flag Count
47 ACK Flag Count
48 URG Flag Count
49 CWE Flag Count
50 ECE Flag Count
51 Down/

In [41]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

Fwd Packet Length Mean         0.066291
Flow IAT Max                   0.066284
Bwd Packets/s                  0.065682
Average Packet Size            0.064058
Subflow Fwd Bytes              0.063421
Total Length of Fwd Packets    0.058454
Fwd Packet Length Max          0.055073
Avg Fwd Segment Size           0.047518
Flow Duration                  0.046930
PSH Flag Count                 0.043320
dtype: float64


In [42]:
X_train2 = X_train.drop(columns=["Destination Port"])
X_test2 = X_test.drop(columns=["Destination Port"])

model2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model2.fit(X_train2, y_train)
print(confusion_matrix(y_test, model2.predict(X_test2)))

[[25458     1]
 [    4 31757]]


In [43]:
print(X_train2.shape)

(228876, 77)


In [44]:
from sklearn.ensemble import IsolationForest

X_benign = X_train[y_train == 0]

iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_benign)

pred = iso.predict(X_test)
y_pred_iso = (pred == -1).astype(int)

print(confusion_matrix(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso, target_names=["BENIGN", "PortScan"]))

[[25186   273]
 [31761     0]]
              precision    recall  f1-score   support

      BENIGN       0.44      0.99      0.61     25459
    PortScan       0.00      0.00      0.00     31761

    accuracy                           0.44     57220
   macro avg       0.22      0.49      0.31     57220
weighted avg       0.20      0.44      0.27     57220



In [45]:
from sklearn.metrics import roc_auc_score

scores = -iso.score_samples(X_test)
print(roc_auc_score(y_test, scores))
print(pd.Series(scores, index=y_test.index).groupby(y_test).median())

0.5592807382300824
Label
0    0.347503
1    0.352176
dtype: float64


In [46]:
ddos = pd.read_csv("../data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
ddos.columns = ddos.columns.str.strip()
ddos = ddos.replace([np.inf, -np.inf], np.nan).dropna()
print(ddos.shape)
print(ddos["Label"].value_counts())

(225711, 79)
Label
DDoS      128025
BENIGN     97686
Name: count, dtype: int64


In [47]:
X_ddos = ddos.drop(columns=["Label"])
y_ddos = (ddos["Label"] == "DDoS").astype(int)

pred_ddos = model.predict(X_ddos)
pd.crosstab(y_ddos, pred_ddos, rownames=["Real"], colnames=["Guess"])

Guess,0,1
Real,,
0,97686,0
1,53857,74168


In [48]:
pred_iso_ddos = iso.predict(X_ddos)
y_pred_iso_ddos = (pred_iso_ddos == -1).astype(int)
pd.crosstab(y_ddos, y_pred_iso_ddos, rownames=["Real"], colnames=["Guess"])

Guess,0,1
Real,,
0,88359,9327
1,106596,21429


## Findings

**Port scans.** The Random Forest made 5 mistakes out of 57,220 test connections, the 20% it never saw during training. Scans were easy to spot because of how they behave, not which port they hit: a scan connection lasts about 47 microseconds and sends about 1 packet, while a normal one lasts about 30,964 microseconds. Removing the destination port changed nothing.

**DDoS, an attack the models never studied.** The Random Forest caught 74,168 of 128,025 attacks (58%) with 0 false alarms. The Isolation Forest caught only 21,429 (17%) and wrongly flagged 9,327 normal connections (9.5%), even though it was set to flag about 1%.

**Why the Isolation Forest failed on port scans.** It flags connections that look unusual next to normal traffic. A single scan connection is tiny and fast, and normal traffic probably contains many connections like that, so on its own it looks normal. A scan only shows as a pattern across many connections, and the model reads one at a time. It caught 0 of 31,761 scans.

**Realistic traffic (attacks at 1%).** On the DDoS file with attacks reduced to 1% of connections, the Random Forest raised 555 alerts, all real, but missed 432 attacks. The Isolation Forest raised 9,498 alerts and only 171 were real (about 2%), so an analyst would be flooded with false alarms.

**Training on all three attack families.** A Random Forest trained on port scan, DDoS and brute force (FTP and SSH), and tested on held-out connections, made 14 mistakes out of 191,491 with no false alarms. It also names the attack type and never confused one attack for another. Before this, a model trained on port scans alone caught 1 of 13,832 brute force attacks, and adding DDoS to its training did not help (0 caught). The detector only recognizes attack families it has been trained on. These scores are probably optimistic, because the test connections come from the same attack runs as the training ones.

**Limits.**
- 55% of this file is attacks. On a real network attacks are a tiny fraction, so these scores will not carry over.
- The models studied one day and one attack type. On a different attack the Random Forest missed 42%.
- Still to test: mostly normal traffic, and my own lab traffic.

In [49]:
normal_idx = y_ddos[y_ddos == 0].index
attack_idx = y_ddos[y_ddos == 1].sample(n=987, random_state=42).index
idx = normal_idx.union(attack_idx)

X_r = X_ddos.loc[idx]
y_r = y_ddos.loc[idx]
print(y_r.shape, y_r.mean())

rf_pred = model.predict(X_r)
iso_pred = (iso.predict(X_r) == -1).astype(int)

print("Random Forest")
print(confusion_matrix(y_r, rf_pred))
print("Isolation Forest")
print(confusion_matrix(y_r, iso_pred))

(98673,) 0.010002736310844912
Random Forest
[[97686     0]
 [  432   555]]
Isolation Forest
[[88359  9327]
 [  816   171]]


In [50]:
tue = pd.read_csv("../data/Tuesday-WorkingHours.pcap_ISCX.csv")
tue.columns = tue.columns.str.strip()
tue = tue.replace([np.inf, -np.inf], np.nan).dropna()
print(tue.shape)
print(tue["Label"].value_counts())

(445645, 79)
Label
BENIGN         431813
FTP-Patator      7935
SSH-Patator      5897
Name: count, dtype: int64


In [51]:
X_tue = tue.drop(columns=["Label"])

rf_pred = model.predict(X_tue)
iso_pred = (iso.predict(X_tue) == -1).astype(int)

print("Random Forest")
print(pd.crosstab(tue["Label"], rf_pred, rownames=["Real"], colnames=["Guess"]))
print("Isolation Forest")
print(pd.crosstab(tue["Label"], iso_pred, rownames=["Real"], colnames=["Guess"]))

Random Forest
Guess             0   1
Real                   
BENIGN       431797  16
FTP-Patator    7935   0
SSH-Patator    5896   1
Isolation Forest
Guess             0     1
Real                     
BENIGN       427784  4029
FTP-Patator    7935     0
SSH-Patator    5897     0


In [52]:
combo = pd.concat([df, ddos], ignore_index=True)
X_combo = combo.drop(columns=["Label"])
y_combo = (combo["Label"] != "BENIGN").astype(int)

rf2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf2.fit(X_combo, y_combo)

rf2_pred = rf2.predict(X_tue)
pd.crosstab(tue["Label"], rf2_pred, rownames=["Real"], colnames=["Guess"])

Guess,0
Real,
BENIGN,431813
FTP-Patator,7935
SSH-Patator,5897


In [53]:
all_df = pd.concat([df, ddos, tue], ignore_index=True)
X_all = all_df.drop(columns=["Label"])
y_all = all_df["Label"]

Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

rf3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf3.fit(Xa_train, ya_train)

rf3_pred = rf3.predict(Xa_test)
pd.crosstab(ya_test, rf3_pred, rownames=["Real"], colnames=["Guess"])

Guess,BENIGN,DDoS,FTP-Patator,PortScan,SSH-Patator
Real,,,,,
BENIGN,131359,0,0,0,0
DDoS,8,25597,0,0,0
FTP-Patator,2,0,1585,0,0
PortScan,2,0,0,31759,0
SSH-Patator,2,0,0,0,1177


In [54]:
import joblib, os

os.makedirs("../models", exist_ok=True)
joblib.dump({"model": rf3, "features": list(X_all.columns)}, "../models/rf3.joblib", compress=3)
print(round(os.path.getsize("../models/rf3.joblib") / 1e6, 1), "MB")

1.6 MB


In [55]:
import sys
sys.path.append("..")
from src.detector import predict

sample = Xa_test.sample(5, random_state=1)
print(predict(sample).assign(real=ya_test.loc[sample.index]))

         label  confidence    real
869124  BENIGN         1.0  BENIGN
287611  BENIGN         1.0  BENIGN
318758  BENIGN         1.0  BENIGN
508769  BENIGN         1.0  BENIGN
532683  BENIGN         1.0  BENIGN


In [56]:
idx = ya_test.groupby(ya_test).sample(2, random_state=1).index
print(predict(Xa_test.loc[idx]).assign(real=ya_test.loc[idx]))

              label  confidence         real
442366       BENIGN         1.0       BENIGN
670334       BENIGN         1.0       BENIGN
375264         DDoS         1.0         DDoS
422077         DDoS         1.0         DDoS
576324  FTP-Patator         1.0  FTP-Patator
577290  FTP-Patator         1.0  FTP-Patator
268860     PortScan         1.0     PortScan
207550     PortScan         1.0     PortScan
841600  SSH-Patator         1.0  SSH-Patator
703861  SSH-Patator         1.0  SSH-Patator


In [57]:
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")
client = Groq()

reply = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
)
print(reply.choices[0].message.content)

ready


In [58]:
for m in client.models.list().data:
    print(m.id)

whisper-large-v3-turbo
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
allam-2-7b
canopylabs/orpheus-v1-english


In [59]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(Xa_train, ya_train)

pred_gb = gb.predict(Xa_test)
print("Gradient boosting errors:", (pred_gb != ya_test).sum(), "out of", len(ya_test))

pred_rf = rf3.predict(Xa_test)
print("Random Forest errors:", (pred_rf != ya_test).sum(), "out of", len(ya_test))

Gradient boosting errors: 7 out of 191491
Random Forest errors: 14 out of 191491


In [60]:
imp = pd.Series(rf3.feature_importances_, index=Xa_train.columns).sort_values(ascending=False)
print(imp.head(10))

Bwd Packets/s                  0.065733
Subflow Fwd Bytes              0.047937
Average Packet Size            0.043791
Fwd Packet Length Max          0.042454
Bwd Packet Length Mean         0.039858
Fwd Packet Length Mean         0.039524
Total Length of Fwd Packets    0.038718
Avg Fwd Segment Size           0.038632
Bwd Packet Length Max          0.033512
Avg Bwd Segment Size           0.032853
dtype: float64


In [61]:
drop = list(imp.head(10).index)

rf_t = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_t.fit(Xa_train.drop(columns=drop), ya_train)

pred_t = rf_t.predict(Xa_test.drop(columns=drop))
print("Errors without the top 10 features:", (pred_t != ya_test).sum(), "out of", len(ya_test))

Errors without the top 10 features: 14 out of 191491


### Shortcut check

The Random Forest's top 10 features are all behavior features (packet sizes and packet
rates). None is a port or an ID-like column. I removed those 10 features and retrained:
the model still made 14 errors out of 191,491, the same as before. So the result does
not depend on a few features that could give the answer away.

This does not prove the model will work on new traffic. My random split can place
near-duplicate attack connections in both train and test sets, which is why brute force,
an attack the model never saw, was missed.

In [62]:
lab_scan = pd.read_csv("../data/lab/scan.csv")
lab_norm = pd.read_csv("../data/lab/normal.csv")
print("lab shapes:", lab_scan.shape, lab_norm.shape)

print("\nLAB port scan, median values:")
print(lab_scan[["flow_duration", "flow_iat_mean", "flow_pkts_s", "tot_fwd_pkts", "tot_bwd_pkts"]].median())

ps = df[df["Label"] == "PortScan"]
print("\nDATASET port scan, median values:")
print(ps[["Flow Duration", "Flow IAT Mean", "Flow Packets/s", "Total Fwd Packets", "Total Backward Packets"]].median())

lab shapes: (1000, 82) (41, 82)

LAB port scan, median values:
flow_duration        0.000137
flow_iat_mean        0.000068
flow_pkts_s      21897.810219
tot_fwd_pkts         2.000000
tot_bwd_pkts         1.000000
dtype: float64

DATASET port scan, median values:
Flow Duration                47.00000
Flow IAT Mean                47.00000
Flow Packets/s            42553.19149
Total Fwd Packets             1.00000
Total Backward Packets        1.00000
dtype: float64


In [63]:
print(list(X_all.columns))

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [64]:
import sys
sys.path.append("..")
from src.lab_features import to_model_features
from src.detector import predict

for name, lab in [("scan", lab_scan), ("normal", lab_norm)]:
    res = predict(to_model_features(lab))
    print(name, "->", res["label"].value_counts().to_dict())

scan -> {'BENIGN': 1000}
normal -> {'BENIGN': 41}


In [65]:
conv = to_model_features(lab_scan)
benign = df[df["Label"] == "BENIGN"]

cmp = pd.DataFrame({
    "lab_scan": conv.median(),
    "dataset_portscan": ps[conv.columns].median(),
    "dataset_benign": benign[conv.columns].median(),
})
rows = list(imp.head(10).index) + ["Init_Win_bytes_forward", "Init_Win_bytes_backward",
                                   "Destination Port", "Total Fwd Packets", "RST Flag Count", "SYN Flag Count"]
print(cmp.loc[rows])

                                lab_scan  dataset_portscan  dataset_benign
Bwd Packets/s                7299.270073       21276.59574       25.731415
Subflow Fwd Bytes             116.000000           0.00000       68.000000
Average Packet Size            58.666667           3.00000       75.500000
Fwd Packet Length Max          58.000000           0.00000       40.000000
Bwd Packet Length Mean         60.000000           6.00000       83.000000
Fwd Packet Length Mean         58.000000           0.00000       37.000000
Total Length of Fwd Packets   116.000000           0.00000       68.000000
Avg Fwd Segment Size           58.000000           0.00000       37.000000
Bwd Packet Length Max          60.000000           6.00000       91.000000
Avg Bwd Segment Size           60.000000           6.00000       83.000000
Init_Win_bytes_forward       1024.000000       29200.00000       57.000000
Init_Win_bytes_backward         0.000000           0.00000       -1.000000
Destination Port         

In [66]:
print(lab_scan[["tot_fwd_pkts", "totlen_fwd_pkts", "fwd_header_len",
                "tot_bwd_pkts", "totlen_bwd_pkts", "bwd_header_len"]].head(5))

   tot_fwd_pkts  totlen_fwd_pkts  fwd_header_len  tot_bwd_pkts  \
0             3              170              60             1   
1             2              116              40             1   
2             2              116              40             1   
3             2              116              40             1   
4             2              116              40             1   

   totlen_bwd_pkts  bwd_header_len  
0               60              20  
1               60              20  
2               60              20  
3               60              20  
4               60              20  


In [67]:
def fix_payload(t):
    t = t.copy()
    per_packet = ["Fwd Packet Length Max", "Fwd Packet Length Min", "Fwd Packet Length Mean",
                  "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean",
                  "Min Packet Length", "Max Packet Length", "Packet Length Mean",
                  "Avg Fwd Segment Size", "Avg Bwd Segment Size", "Average Packet Size"]
    for c in per_packet:
        t[c] = (t[c] - 54).clip(lower=0)
    t["Total Length of Fwd Packets"] = (t["Total Length of Fwd Packets"] - 54 * t["Total Fwd Packets"]).clip(lower=0)
    t["Total Length of Bwd Packets"] = (t["Total Length of Bwd Packets"] - 54 * t["Total Backward Packets"]).clip(lower=0)
    t["Subflow Fwd Bytes"] = t["Total Length of Fwd Packets"]
    t["Subflow Bwd Bytes"] = t["Total Length of Bwd Packets"]
    return t

def count(t):
    return predict(t)["label"].value_counts().to_dict()

scan_fixed = fix_payload(to_model_features(lab_scan))
norm_fixed = fix_payload(to_model_features(lab_norm))
print("1) packet size fixed, scan:  ", count(scan_fixed))
print("1) packet size fixed, normal:", count(norm_fixed))

scan_w = scan_fixed.copy()
scan_w["Init_Win_bytes_forward"] = 29200
print("2) size fixed + window 29200, scan:", count(scan_w))

1) packet size fixed, scan:   {'BENIGN': 1000}
1) packet size fixed, normal: {'BENIGN': 41}
2) size fixed + window 29200, scan: {'BENIGN': 1000}


In [68]:
lab_mid = scan_fixed.median()
ref_mid = ps[lab_mid.index].median()
spread = ps[lab_mid.index].std().replace(0, 1)
gap = (lab_mid - ref_mid).abs() / spread
score = (gap * imp[lab_mid.index]).sort_values(ascending=False)

out = pd.DataFrame({"lab": lab_mid, "dataset_scan": ref_mid, "score": score})
print(out.loc[score.head(10).index])

                                      lab  dataset_scan     score
PSH Flag Count               0.000000e+00        1.0000  1.525046
ACK Flag Count               1.000000e+00        0.0000  1.156650
Fwd Packet Length Mean       4.000000e+00        0.0000  0.120601
Avg Fwd Segment Size         4.000000e+00        0.0000  0.117879
Subflow Fwd Bytes            8.000000e+00        0.0000  0.067712
Total Length of Fwd Packets  8.000000e+00        0.0000  0.054689
Init_Win_bytes_forward       1.024000e+03    29200.0000  0.051208
Fwd Packet Length Max        4.000000e+00        0.0000  0.048183
Flow Bytes/s                 1.284672e+06   139534.8837  0.042846
Total Fwd Packets            2.000000e+00        1.0000  0.031448


In [69]:
t3 = scan_fixed.copy()
t3["PSH Flag Count"] = 1
t3["ACK Flag Count"] = 0
print("3) scan with PSH=1, ACK=0:", count(t3))

t4 = t3.copy()
t4["Init_Win_bytes_forward"] = 29200
print("4) ...and window 29200:   ", count(t4))

n3 = norm_fixed.copy()
n3["PSH Flag Count"] = 1
n3["ACK Flag Count"] = 0
print("5) normal with PSH=1, ACK=0:", count(n3))

3) scan with PSH=1, ACK=0: {'BENIGN': 946, 'PortScan': 54}
4) ...and window 29200:    {'BENIGN': 981, 'PortScan': 19}
5) normal with PSH=1, ACK=0: {'BENIGN': 41}


### Test on my own lab traffic

I ran an nmap port scan from Kali against an Ubuntu VM and captured normal web traffic
between the same two machines, then converted both captures to flows with a Python
version of CICFlowMeter. The model labeled all 1,000 scan flows as BENIGN (0 caught)
and all 41 normal flows correctly. So it had no false alarms, but it missed the scan.

Causes I found: (1) my tool measures packet size with the packet headers included,
while the dataset counts only the content, which I corrected approximately; (2) the
dataset's scan flows have PSH=1 and ACK=0 flag counts, which a plain scan packet does
not produce, so the original tool probably counts flags differently, and the model
learned that quirk. Fixing the flags made the model catch 54 of 1,000 scan flows.
The rest of the gap comes from many small differences.

Correction to my earlier shortcut check: removing the top 10 features did not change
the result, but that only shows the model has backup features. The flag counts were
not in the top 10 and still acted as a shortcut.

Conclusion: a model trained on CIC-IDS2017 does not transfer to traffic made with
different tools. Next step: add my own lab flows to the training data and test on a
second scan I did not train on.

In [70]:
import importlib, src.lab_features as lf
importlib.reload(lf)

def load_lab(name):
    t = lf.fix_payload(lf.to_model_features(pd.read_csv(f"../data/lab/{name}.csv")))
    return t.replace([np.inf, -np.inf], np.nan).fillna(0)

scan1, norm1, scan2, norm2 = [load_lab(n) for n in ["scan", "normal", "scan2", "normal2"]]
print("lab flows:", len(scan1), len(norm1), len(scan2), len(norm2))

X_tr = pd.concat([Xa_train, scan1, norm2], ignore_index=True)[Xa_train.columns]
y_tr = pd.concat([ya_train, pd.Series(["PortScan"] * len(scan1)), pd.Series(["BENIGN"] * len(norm2))], ignore_index=True)

rf4 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf4.fit(X_tr, y_tr)

for name, data in [("scan2 (unseen scan)", scan2), ("normal1 (simple web)", norm1)]:
    print(name, "-> old model:", pd.Series(rf3.predict(data[Xa_train.columns])).value_counts().to_dict(),
          "| new model:", pd.Series(rf4.predict(data[Xa_train.columns])).value_counts().to_dict())

lab flows: 1000 41 1001 79
scan2 (unseen scan) -> old model: {'BENIGN': 1001} | new model: {'BENIGN': 1001}
normal1 (simple web) -> old model: {'BENIGN': 41} | new model: {'BENIGN': 41}


In [71]:
cols = Xa_train.columns

# 1) Does the new model even recognize the scan flows it was trained on?
print("scan1 (in its training data):", pd.Series(rf4.predict(scan1[cols])).value_counts().to_dict())

# 2) In the dataset's training rows, what are the look-alikes of a tiny scan flow?
sim = Xa_train[(Xa_train["Total Fwd Packets"] <= 2) &
               (Xa_train["Total Backward Packets"] == 1) &
               (Xa_train["Flow Duration"] < 1000)]
print("dataset look-alikes by label:", ya_train.loc[sim.index].value_counts().to_dict())

scan1 (in its training data): {'PortScan': 1000}
dataset look-alikes by label: {'PortScan': 126262, 'BENIGN': 53277, 'SSH-Patator': 2315, 'FTP-Patator': 1959, 'DDoS': 63}


In [72]:
mid1, mid2 = scan1.median(), scan2.median()
spread = scan1.std().replace(0, 1)
score = ((mid2 - mid1).abs() / spread * imp[mid1.index]).sort_values(ascending=False)

out = pd.DataFrame({"scan1_-sS": mid1, "scan2_-sT": mid2, "score": score})
print(out.loc[score.head(8).index])

                          scan1_-sS     scan2_-sT        score
Init_Win_bytes_forward  1024.000000  64240.000000  1620.675358
Fwd Packet Length Mean     4.000000     20.000000     6.138347
Avg Fwd Segment Size       4.000000     20.000000     5.999805
Average Packet Size        4.666667     15.333333     5.181794
Packet Length Variance     0.888889     43.555556     4.128555
Packet Length Mean         4.666667     15.333333     3.742212
Subflow Fwd Bytes          8.000000     40.000000     1.533996
Packet Length Std          0.942809      6.599663     1.329773


In [73]:
def try_drop(drop):
    keep = [c for c in cols if c not in drop]
    m = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr[keep], y_tr)
    return {
        "removed": len(drop),
        "scan2": pd.Series(m.predict(scan2[keep])).value_counts().to_dict(),
        "normal1": pd.Series(m.predict(norm1[keep])).value_counts().to_dict(),
        "dataset errors": int((m.predict(Xa_test[keep]) != ya_test).sum()),
    }

drop_win = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]
size_words = ["Length", "Segment Size", "Packet Size", "Subflow", "Bytes/s", "min_seg_size"]
drop_size = drop_win + [c for c in cols if any(w in c for w in size_words) and c not in drop_win]

print("A (window removed):        ", try_drop(drop_win))
print("B (window + sizes removed):", try_drop(drop_size))

A (window removed):         {'removed': 2, 'scan2': {'BENIGN': 1001}, 'normal1': {'BENIGN': 41}, 'dataset errors': 35}
B (window + sizes removed): {'removed': 29, 'scan2': {'PortScan': 1000, 'BENIGN': 1}, 'normal1': {'BENIGN': 41}, 'dataset errors': 75}


In [74]:
import joblib

keep = [c for c in cols if c not in drop_size]
X_final = pd.concat([X_all, scan1, scan2, norm1, norm2], ignore_index=True)[keep]
y_final = pd.concat([y_all,
                     pd.Series(["PortScan"] * (len(scan1) + len(scan2))),
                     pd.Series(["BENIGN"] * (len(norm1) + len(norm2)))],
                    ignore_index=True)

rf5 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf5.fit(X_final, y_final)

joblib.dump({"model": rf5, "features": keep}, "../models/rf5.joblib", compress=3)
print("saved with", len(keep), "features")

saved with 49 features


In [75]:
import importlib, src.detector as det
importlib.reload(det)

for name, data in [("scan2", scan2), ("scan1", scan1), ("normal1", norm1), ("normal2", norm2)]:
    res = det.predict(data)
    print(name, res["label"].value_counts().to_dict(), "avg confidence:", round(res["confidence"].mean(), 2))

scan2 {'BENIGN': 1001} avg confidence: 0.88
scan1 {'BENIGN': 1000} avg confidence: 0.74
normal1 {'BENIGN': 41} avg confidence: 0.63
normal2 {'BENIGN': 79} avg confidence: 0.65


In [76]:
print(det.MODEL_PATH)
print(len(det._features), "features")

C:\Users\DELL\network-alert-triage\models\rf3.joblib
78 features


In [77]:
import importlib, src.detector as det
importlib.reload(det)
print(det.MODEL_PATH.name, len(det._features), "features")

for name, data in [("scan2", scan2), ("scan1", scan1), ("normal1", norm1), ("normal2", norm2)]:
    res = det.predict(data)
    print(name, res["label"].value_counts().to_dict(), "avg confidence:", round(res["confidence"].mean(), 2))

rf5.joblib 49 features
scan2 {'PortScan': 1001} avg confidence: 1.0
scan1 {'PortScan': 1000} avg confidence: 1.0
normal1 {'BENIGN': 41} avg confidence: 1.0
normal2 {'BENIGN': 79} avg confidence: 1.0


### Making the model work on my own lab traffic

The model first missed every scan flow from my lab. Training it on my first scan
(nmap -sS) let it recognize that scan, but it still missed a second scan made with a
different mode (nmap -sT). The top differences were the TCP window size in the first
packet (1,024 vs 64,240) and the packet sizes: these depend on the tool that builds
the packet, not on scanning behavior, so the model had memorized the first tool's
fingerprint.

I retrained without the window-size columns and without all packet-size columns
(29 of 78 columns). Removing only the window was not enough. With both removed, the
model caught 1,000 of the 1,001 flows of the unseen second scan and kept all 41 normal
flows correct. The cost on the original dataset test set was small: 75 errors out of
191,491 instead of 14.

The final model uses the remaining 49 columns (timing, packet counts, flags, rates)
and is trained on the dataset plus all my lab flows.

Limits: one lab, one attacker, one victim, one attack type. The second scan is the
same attack with a different setting, not a different attack. The normal-traffic test
had only 41 simple flows, so the false-alarm rate is not well measured.